# Quickstart: The Core Pipeline
CodeGraphene bridges the gap between static analysis and Large Language Models (LLMs).
It does this in three steps:
1. **Parse**: Convert raw source code into a mathematical Code Property Graph (CPG).
2. **Trim**: Extract only the subgraph that is relevant to a specific target.
3. **Serialize**: Flatten that subgraph back into a clean text prompt for an LLM.

## Setup and Imports
We will be analyzing `sample_code.py`, which contains a snippet from the RepoCoder `function_level_completion_4k_context_codex` dataset.

In [ ]:
from codegraphene.core import NodeGranularity
from codegraphene.parsers.joern import JoernParser
from codegraphene.trimmers.khop import KHopTrimmer
from codegraphene.serializers.text import CodeReconstructionSerializer
from codegraphene.pipeline import GraphPipeline

target_file = "sample_code.py"

## Configuring the Pipeline
In keeping with the three steps mentioned above, we build our GraphPipeline by defining a parser, a trimmer, and a serializer. 

In [2]:
# Parser: JoernParser, which uses Joern under the hood.
parser = JoernParser(granularity=NodeGranularity.LINE)

# Trimmer: A simple K-hop trimmer. The number of hops can be configured. Here, we use hops=1 for illustrative purposes.
trimmer = KHopTrimmer(hops=1)

# Serializer: CodeReconstructionSerializer, which turns our trimmer subgraph back into code.
serializer = CodeReconstructionSerializer(granularity=NodeGranularity.LINE)

pipeline = GraphPipeline(
    parser=parser,
    trimmer=trimmer,
    serializer=serializer
)

## Running the pipeline
In this example, we defined our pipeline components to function at line-level granularity (see `01_granularities.ipynb` for more information). We define a target line and run the pipeline on our file while targeting that line.

In [4]:
target_line = 63
output = pipeline.run(file_path=target_file, target=target_line)

print("\n--- FINAL PROMPT ---")
print(output)

[Pipeline] Parsing sample_code.py...
[JoernParser] Parsing source code at: sample_code.py
[JoernParser] Running: joern-parse sample_code.py --output /tmp/tmpqa0kocj8/cpg.bin
[JoernParser] Running: joern-export /tmp/tmpqa0kocj8/cpg.bin --repr all --out /tmp/tmpqa0kocj8/export
[JoernParser] Ingesting DOT file into NetworkX...
[Pipeline] Trimming graph around 63 (Node 25769803790)...
[Pipeline] Trimmed from 316 to 6 nodes.
[Pipeline] Serializing subgraph...

--- FINAL PROMPT ---
Line 63: tmp5 = tmp6 = range(len(self._children))
tmp6.__iter__()
